# MOHIM motif extraction and inspection

다운로드된 원곡에서 Demucs stem을 만들고, stem별 motif 후보를 비교해 선택 결과를 직접 듣는 Colab 노트북입니다. YouTube 다운로드, manifest, ACE-Step, LoRA 학습은 수행하지 않습니다.

## 0. Drive 마운트와 저장소 준비

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import subprocess

REPOSITORY = 'https://github.com/youhan200203/MOHIM.git'
BRANCH = 'working'
REPO_DIR = Path('/content/MOHIM')

if not (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', 'clone', '-b', BRANCH, REPOSITORY, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'switch', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
print('repository:', REPO_DIR)

In [ ]:
%pip install -q -r requirements.txt
%pip install -q --no-deps "beat-this @ git+https://github.com/CPJKU/beat_this.git"

# 첫 설치 후 import 오류가 나면 런타임을 한 번 재시작하고 이 셀부터 다시 실행하세요.

## 1. 경로, 버전, 처리 범위 설정

In [ ]:
from pathlib import Path
import shutil

MOTIF_DATASET_VERSION = 'joint_top4_v1'
MAX_SONGS = 10
MAX_TRACK_DURATION = 300.0
DEVICE = 'cuda'
AUDIO_FORMAT = 'flac'
MOTIF_BARS = 4
MOTIF_SIMILARITY = 0.56
MOTIF_SEARCH_SECONDS = 30.0

DATASET_DIR = Path('/content/drive/MyDrive/MOHIM/genius_pop_dataset')
TRACKS_JSON = DATASET_DIR / 'tracks.json'
AUDIO_DIR = DATASET_DIR / 'audio'
OUTPUT_DIR = Path('/content/drive/MyDrive/MOHIM/motif_dataset')
MOTIF_VERSION_PATH = OUTPUT_DIR / '.mohim_motif_version'
BEAT_CHECKPOINT = Path('/content/checkpoints/beat_this_final0.ckpt')

assert TRACKS_JSON.is_file(), f'tracks.json이 없습니다: {TRACKS_JSON}'
stored_version = (
    MOTIF_VERSION_PATH.read_text(encoding='utf-8').strip()
    if MOTIF_VERSION_PATH.is_file() else None
)
if stored_version != MOTIF_DATASET_VERSION:
    print(f'motif dataset reset: {stored_version!r} -> {MOTIF_DATASET_VERSION!r}')
    if OUTPUT_DIR.exists():
        shutil.rmtree(OUTPUT_DIR)
    OUTPUT_DIR.mkdir(parents=True)
    MOTIF_VERSION_PATH.write_text(MOTIF_DATASET_VERSION + '\n', encoding='utf-8')
else:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print('motif dataset version unchanged; completed songs will be skipped')
print('audio:', AUDIO_DIR)
print('motif dataset:', OUTPUT_DIR)
print('maximum songs:', MAX_SONGS)

## 2. 기존 다운로드 음원에서 처리할 곡 선택

In [ ]:
import pandas as pd
from mohim.local_dataset import index_audio_files, load_local_tracks, resolve_audio_path

tracks = load_local_tracks(
    TRACKS_JSON,
    require_lyrics=True,
    max_duration_seconds=MAX_TRACK_DURATION,
)
audio_index = index_audio_files(AUDIO_DIR)
matched = [(track, resolve_audio_path(track, audio_index)) for track in tracks]
matched = [(track, path) for track, path in matched if path is not None][:MAX_SONGS]
assert matched, '처리할 다운로드 음원이 없습니다.'
display(pd.DataFrame([
    {
        'track_id': track.track_id, 'artist': track.artist, 'title': track.title,
        'duration_seconds': track.duration_seconds, 'audio_path': str(path),
    }
    for track, path in matched
]))

## 3. Demucs와 motif extractor 준비

In [ ]:
import urllib.request
from mohim.motif import MotifConfig, MotifExtractor, create_beat_tracker
from mohim.separator import StemSeparator

BEAT_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
if not BEAT_CHECKPOINT.is_file():
    urllib.request.urlretrieve(
        'https://cloud.cp.jku.at/public.php/dav/files/7ik4RrBKTS273gp/final0.ckpt',
        BEAT_CHECKPOINT,
    )
separator = StemSeparator(device=DEVICE, model_name='htdemucs_6s')
beat_tracker = create_beat_tracker(BEAT_CHECKPOINT, device=DEVICE)
motif_extractor = MotifExtractor(
    beat_tracker,
    MotifConfig(
        bars=MOTIF_BARS,
        search_seconds=MOTIF_SEARCH_SECONDS,
        similarity_threshold=MOTIF_SIMILARITY,
    ),
)
print('separator and motif extractor ready')

## 4. 최대 10곡 Demucs·motif 처리

버전이 같은 상태에서 재실행하면 완료된 곡은 건너뛰고 중단 지점부터 이어갑니다.

In [ ]:
from dataclasses import asdict
from mohim.dataset import DatasetBuilder

builder = DatasetBuilder(
    audio_dir=AUDIO_DIR,
    output_dir=OUTPUT_DIR,
    separator=separator,
    motif_extractor=motif_extractor,
    audio_format=AUDIO_FORMAT,
    resume=True,
)
results = builder.build([track for track, _ in matched], max_songs=MAX_SONGS)
results_df = pd.DataFrame([asdict(result) for result in results])
display(results_df)
display(results_df.groupby(['status', 'reason'], dropna=False).size().rename('count').reset_index())

## 5. 곡별 motif, stem, 점수 듣기

`DEBUG_TRACK_INDEX`를 바꿔 처리된 곡을 하나씩 확인합니다. drums는 개별 저장하지 않지만 accompaniment에는 포함됩니다.

In [ ]:
import json
from IPython.display import Audio, display

DEBUG_TRACK_INDEX = 0
metadata_paths = sorted(OUTPUT_DIR.glob('*/metadata.json'))
assert metadata_paths, '완료된 motif 샘플이 없습니다.'
assert 0 <= DEBUG_TRACK_INDEX < len(metadata_paths), 'DEBUG_TRACK_INDEX 범위를 확인하세요.'
metadata_path = metadata_paths[DEBUG_TRACK_INDEX]
sample_dir = metadata_path.parent
metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
print(metadata['track_id'], metadata.get('artist'), '-', metadata.get('title'))
print('selected stem:', metadata['motif_stem'])
print('segment:', metadata['motif_start_sec'], '~', metadata['motif_end_sec'])
display(pd.DataFrame(metadata['motif_scores']).T.sort_values('total', ascending=False))

listen_files = [('selected motif', metadata['motif_seed_file'])]
for stem_name, filename in metadata.get('stem_files', {}).items():
    listen_files.append((stem_name, filename))
listen_files.append(('accompaniment', metadata['accompaniment_target_file']))
for label, filename in listen_files:
    path = sample_dir / filename
    if path.is_file():
        print('---', label, '---', path.name)
        display(Audio(filename=str(path)))